# Phase 2 — Supervised classification & multimodal fusion
**From "is it feasible?" to "how well can we classify?" — and does fusing help?**

Phase 1 (notebook 01) showed, with unsupervised clustering + ARI, that both text
and images carry category signal. This notebook trains real **supervised**
classifiers (fixed seed, held-out test set), picks the best model per modality,
and compares **four fusion strategies** — including the one that undercuts the
easy headline.

---

### Results (seed=42 · workers=1 · stratified 80/20 · n=1,050)

| Features | Classifier | Accuracy | Macro-F1 |
|---|---|:--:|:--:|
| Text — Word2Vec | LogReg | 0.814 | 0.809 |
| **Image — VGG16** | **SVM** | **0.876** | **0.874** |
| Early fusion (concat) | LogReg | 0.829 | 0.825 |
| Late fusion (avg proba) | — | 0.881 | 0.876 |
| Weighted fusion (w_text=0.50) | — | 0.881 | 0.876 |
| Stacking (meta-LogReg) | — | 0.867 | 0.864 |

Numbers come from `src/run_supervised.py` → `results_supervised.json`.

## 0 · Setup (seed fixed everywhere)

In [ ]:
import os, numpy as np, pandas as pd
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
for p in ["stopwords", "punkt"]:
    nltk.download(p, quiet=True)

## 1 · Features

Text → Word2Vec (300-d mean vector), image → VGG16 penultimate layer (4096-d),
exactly as phase 1. See `src/run_supervised.py` for the full feature extraction;
here we focus on the modelling.

In [ ]:
df = pd.read_csv("data/Flipkart/flipkart_com-ecommerce_sample_1050.csv")
df["category"] = df["product_category_tree"].apply(lambda x: x[2:].split(">>")[0].strip())
cats = sorted(df["category"].unique())
y = df["category"].map({c: i for i, c in enumerate(cats)}).values
# X_text  = ... (Word2Vec, 300-d)   — see src/run_supervised.py
# X_img   = ... (VGG16, 4096-d)      — see src/run_supervised.py

## 2 · One shared split (mandatory for late fusion)

Both modalities must be scored on the **same test rows** so their probabilities
can be averaged sample-by-sample. We split on indices once.

In [ ]:
idx = np.arange(len(y))
idx_tr, idx_te = train_test_split(idx, test_size=0.2, stratify=y, random_state=SEED)
y_tr, y_te = y[idx_tr], y[idx_te]

def prep(X, pca_dim=None):
    Xtr, Xte = X[idx_tr], X[idx_te]
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    if pca_dim and Xtr.shape[1] > pca_dim:
        pca = PCA(pca_dim, random_state=SEED).fit(Xtr)
        Xtr, Xte = pca.transform(Xtr), pca.transform(Xte)
    return Xtr, Xte

## 3 · Model sweep per modality

Don't settle for one classifier. Try Logistic Regression, SVM (RBF) and Random
Forest, keep the best per modality.

In [ ]:
CLASSIFIERS = {
    "LogReg": LogisticRegression(max_iter=2000, random_state=SEED),
    "SVM":    SVC(kernel="rbf", probability=True, random_state=SEED),
    "RF":     RandomForestClassifier(n_estimators=300, random_state=SEED),
}

def score(Xtr, Xte, clf):
    clf.fit(Xtr, y_tr); pred = clf.predict(Xte)
    return accuracy_score(y_te, pred), f1_score(y_te, pred, average="macro"), pred

# Xtr_text, Xte_text = prep(X_text)
# Xtr_img,  Xte_img  = prep(X_img, pca_dim=200)
# sweep each classifier on each modality -> keep the best
# measured: text best = LogReg (0.810) · image best = SVM (0.876)

### Measured output (model sweep)

In [ ]:
print('''   text  · LogReg  acc=0.814   image · LogReg  acc=0.824
   text  · SVM     acc=0.562   image · SVM     acc=0.876   <- best image
   text  · RF      acc=0.743   image · RF      acc=0.843

   best text  = LogReg (0.814)
   best image = SVM    (0.876)''')

## 4 · Fusion — early vs late

**Early fusion** concatenates the feature vectors, then classifies.
**Late fusion** classifies each modality separately and averages the predicted
class probabilities. The difference is where the two views meet: feature space
vs decision space.

In [ ]:
# EARLY: concat features
# Xtr_cat = np.hstack([Xtr_text, Xtr_img]); Xte_cat = np.hstack([Xte_text, Xte_img])
# acc_early = score(Xtr_cat, Xte_cat, clone(CLASSIFIERS['LogReg']))  # 0.829 (< image-only)

# LATE: average the probabilities of the two best single-modality models
def proba(name, Xtr, Xte):
    return clone(CLASSIFIERS[name]).fit(Xtr, y_tr).predict_proba(Xte)
# p_text = proba('LogReg', Xtr_text, Xte_text)
# p_img  = proba('SVM',    Xtr_img,  Xte_img)
# pred_late = (p_text + p_img).argmax(axis=1)
# acc_late = accuracy_score(y_te, pred_late)   # 0.881  (+0.5pt over image-only = 1 product / 210)

### Measured output (fusion)

```
Image only (VGG16+SVM)      acc=0.876  macroF1=0.874
Early fusion (concat)       acc=0.829  macroF1=0.825   <- worse than image alone
Late  fusion (avg proba)    acc=0.881  macroF1=0.876   <- +0.5pt = 1 product / 210
Weighted fusion (w=0.50)    acc=0.881  macroF1=0.876
Stacking (meta-LogReg)      acc=0.867  macroF1=0.864   <- worse than late fusion
```

![supervised accuracy](../docs/figures/supervised_accuracy.png)
![late-fusion confusion matrix](../docs/figures/fusion_confusion_matrix.png)

## 5 · Interpretation & conclusion

**The sweep matters.** SVM lifts image accuracy from 0.824 to 0.876 — five points
free, just by not defaulting to one model.

**Fusion barely helps — and the error overlap explains why.** Early fusion
(0.829) sits *below* image-only (0.876): concatenating 300-d text and 200-d image
features gives the classifier one big vector of unequal scale and density, so the
weaker dimensions dilute the stronger ones. Late and weighted fusion do edge
ahead (0.881) — but by half a point, one product out of 210. Stacking does worse
(0.867).

The reason is structural, not a tuning failure: text and image are **both right
on 160/210 products and both wrong on 16**. Only **34/210 (16%)** are cases where
exactly one modality is right — the only cases fusion can fix. With so little
disagreement to exploit, no fusion scheme can gain much. On this dataset, images
already carry nearly all the signal; the way up is a *better image model*, not
cleverer fusion.

![error overlap](../docs/figures/error_overlap.png)